In [0]:

CATALOG = "workspace"
SCHEMA = "default"
VOLUME = "inventory_vol"

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

INPUT_DIR = VOLUME_PATH
OUTPUT_DIR = f"{VOLUME_PATH}/gold"

TABLE_NAME = f"{CATALOG}.{SCHEMA}.master_inventory"

PRODUCT_FILE = f"{INPUT_DIR}/products.csv"
STOCK_FILE = f"{INPUT_DIR}/stock_movements_clean.csv"
SUPPLIER_FILE = f"{INPUT_DIR}/suppliers.csv"
WAREHOUSE_FILE = f"{INPUT_DIR}/warehouses.csv"

print("Input folder :", INPUT_DIR)
print("Output folder:", OUTPUT_DIR)
print("Delta table  :", TABLE_NAME)

print("\nInput files:")
print(PRODUCT_FILE)
print(STOCK_FILE)
print(SUPPLIER_FILE)
print(WAREHOUSE_FILE)


Input folder : /Volumes/workspace/default/inventory_vol
Output folder: /Volumes/workspace/default/inventory_vol/gold
Delta table  : workspace.default.master_inventory

Input files:
/Volumes/workspace/default/inventory_vol/products.csv
/Volumes/workspace/default/inventory_vol/stock_movements_clean.csv
/Volumes/workspace/default/inventory_vol/suppliers.csv
/Volumes/workspace/default/inventory_vol/warehouses.csv


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

movement_schema = StructType([
    StructField("movement_id",   IntegerType(), False),
    StructField("product_id",    IntegerType(), False),
    StructField("warehouse_id",  IntegerType(), False),
    StructField("movement_type", StringType(),  True),
    StructField("quantity",      IntegerType(), False),   # signed: IN +, OUT -, ADJUSTMENT +/-
    StructField("movement_date", DateType(),    True),
    StructField("reference_no",  StringType(),  True),
    StructField("reason",        StringType(),  True),
])

movements  = spark.read.csv(f"{INPUT_DIR}/stock_movements_clean.csv", header=True, schema=movement_schema)
products   = spark.read.csv(f"{INPUT_DIR}/products.csv",   header=True, inferSchema=True)
warehouses = spark.read.csv(f"{INPUT_DIR}/warehouses.csv", header=True, inferSchema=True)
suppliers  = spark.read.csv(f"{INPUT_DIR}/suppliers.csv",  header=True, inferSchema=True)

print(f"stock movements : {movements.count():>6,} rows")
print(f"products        : {products.count():>6,} rows")
print(f"warehouses      : {warehouses.count():>6,} rows")
print(f"suppliers       : {suppliers.count():>6,} rows")
display(movements.limit(5))

stock movements :  3,813 rows
products        :     48 rows
warehouses      :      5 rows
suppliers       :      8 rows


movement_id,product_id,warehouse_id,movement_type,quantity,movement_date,reference_no,reason
1,1,1,IN,93,2026-01-13,OPEN-1-1,Opening stock
2,1,2,IN,141,2026-01-13,OPEN-1-2,Opening stock
3,1,3,IN,130,2026-01-13,OPEN-1-3,Opening stock
4,1,4,IN,125,2026-01-13,OPEN-1-4,Opening stock
5,1,5,IN,137,2026-01-13,OPEN-1-5,Opening stock


In [0]:
null_counts = movements.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
                                for c in ["movement_id", "product_id", "warehouse_id", "quantity", "movement_date"]])
display(null_counts)

orphan_products   = movements.join(products,   "product_id",   "left_anti").count()
orphan_warehouses = movements.join(warehouses, "warehouse_id", "left_anti").count()
duplicate_ids     = movements.count() - movements.select("movement_id").distinct().count()

print("movements with unknown product   :", orphan_products)
print("movements with unknown warehouse :", orphan_warehouses)
print("duplicate movement ids           :", duplicate_ids)
assert orphan_products == 0 and orphan_warehouses == 0 and duplicate_ids == 0, "Data is not clean - check Week 2 output"
print("Data quality checks passed")

movement_id,product_id,warehouse_id,quantity,movement_date
0,0,0,0,0


movements with unknown product   : 0
movements with unknown warehouse : 0
duplicate movement ids           : 0
Data quality checks passed


In [0]:
stock = movements.groupBy("product_id", "warehouse_id").agg(
    F.sum("quantity").alias("quantity_on_hand"),
    F.max("movement_date").alias("last_movement_date"))

print("product-warehouse combinations:", stock.count())
display(stock.orderBy("warehouse_id", "product_id").limit(10))

product-warehouse combinations: 240


product_id,warehouse_id,quantity_on_hand,last_movement_date
1,1,55,2026-10-08
2,1,166,2026-10-09
3,1,2,2026-10-10
4,1,146,2026-10-07
5,1,8,2026-10-07
6,1,768,2026-10-09
7,1,309,2026-10-07
8,1,286,2026-10-09
9,1,421,2026-10-09
10,1,362,2026-10-07


In [0]:
master = (stock
    .join(products,   "product_id")
    .join(warehouses, "warehouse_id")
    .join(suppliers.select("supplier_id", "supplier_name"), "supplier_id", "left")
    .withColumn("stock_status",
                F.when(F.col("quantity_on_hand") < F.col("reorder_level"), "LOW_STOCK")
                 .when(F.col("quantity_on_hand") > F.col("max_stock_level"), "OVERSTOCK")
                 .otherwise("OK"))
    .withColumn("reorder_flag", F.col("quantity_on_hand") < F.col("reorder_level"))
    .withColumn("shortage_qty", F.greatest(F.col("reorder_level") - F.col("quantity_on_hand"), F.lit(0)))
    .withColumn("excess_qty",   F.greatest(F.col("quantity_on_hand") - F.col("max_stock_level"), F.lit(0)))
    .withColumn("suggested_order_qty", F.when(F.col("reorder_flag"), F.col("reorder_quantity")).otherwise(F.lit(0)))
    .withColumn("stock_value", F.round(F.col("quantity_on_hand") * F.col("unit_price"), 2))
    .withColumn("days_since_last_movement", F.datediff(F.current_date(), F.col("last_movement_date")))
    .select("product_id", "sku", "product_name", "category", "supplier_name",
            "warehouse_id", "warehouse_name", "city",
            "quantity_on_hand", "reorder_level", "max_stock_level", "reorder_quantity",
            "stock_status", "reorder_flag", "shortage_qty", "excess_qty", "suggested_order_qty",
            "unit_price", "stock_value", "last_movement_date", "days_since_last_movement")
    .orderBy("warehouse_id", "product_id"))

print("master inventory rows:", master.count())
display(master.limit(10))

master inventory rows: 240


product_id,sku,product_name,category,supplier_name,warehouse_id,warehouse_name,city,quantity_on_hand,reorder_level,max_stock_level,reorder_quantity,stock_status,reorder_flag,shortage_qty,excess_qty,suggested_order_qty,unit_price,stock_value,last_movement_date,days_since_last_movement
1,ELE-0001,Wireless Mouse,Electronics,Deccan Wholesale Co,1,Chennai Central DC,Chennai,55,25,150,100,OK,false,0,0,0,699,38445,2026-10-08,37
2,ELE-0002,USB-C Cable 1m,Electronics,Deccan Wholesale Co,1,Chennai Central DC,Chennai,166,65,325,210,OK,false,0,0,0,299,49634,2026-10-09,36
3,ELE-0003,Bluetooth Speaker,Electronics,Namma Tech Components,1,Chennai Central DC,Chennai,2,25,150,100,LOW_STOCK,true,23,0,100,1999,3998,2026-10-10,35
4,ELE-0004,Power Bank 10000mAh,Electronics,Deccan Wholesale Co,1,Chennai Central DC,Chennai,146,25,125,80,OVERSTOCK,false,0,21,0,1299,189654,2026-10-07,38
5,ELE-0005,HDMI Cable 2m,Electronics,Deccan Wholesale Co,1,Chennai Central DC,Chennai,8,95,570,380,LOW_STOCK,true,87,0,380,399,3192,2026-10-07,38
6,ELE-0006,Webcam 1080p,Electronics,Namma Tech Components,1,Chennai Central DC,Chennai,768,100,500,320,OVERSTOCK,false,0,268,0,2499,1919232,2026-10-09,36
7,ELE-0007,Laptop Stand,Electronics,Deccan Wholesale Co,1,Chennai Central DC,Chennai,309,105,525,340,OK,false,0,0,0,899,277791,2026-10-07,38
8,ELE-0008,USB Flash Drive 64GB,Electronics,Deccan Wholesale Co,1,Chennai Central DC,Chennai,286,55,220,130,OVERSTOCK,false,0,66,0,549,157014,2026-10-09,36
9,STA-0009,A4 Paper Ream 500 Sheets,Stationery,Southern Star Stationers,1,Chennai Central DC,Chennai,421,100,500,320,OK,false,0,0,0,349,146929,2026-10-09,36
10,STA-0010,Ball Pen Box of 50,Stationery,Southern Star Stationers,1,Chennai Central DC,Chennai,362,105,525,340,OK,false,0,0,0,450,162900,2026-10-07,38


In [0]:
master.createOrReplaceTempView("master_inventory")

display(spark.sql("""
    SELECT stock_status, reorder_flag, COUNT(*) AS product_lines
    FROM master_inventory
    GROUP BY stock_status, reorder_flag
    ORDER BY stock_status
"""))

reorder_list = master.filter("reorder_flag = true").orderBy(F.desc("shortage_qty"))
print("Product lines to reorder:", reorder_list.count())
display(reorder_list.select("sku", "product_name", "warehouse_name", "quantity_on_hand",
                            "reorder_level", "shortage_qty", "suggested_order_qty", "supplier_name").limit(15))

stock_status,reorder_flag,product_lines
LOW_STOCK,true,36
OK,false,153
OVERSTOCK,false,51


Product lines to reorder: 36


sku,product_name,warehouse_name,quantity_on_hand,reorder_level,shortage_qty,suggested_order_qty,supplier_name
STA-0010,Ball Pen Box of 50,Hyderabad South Hub,10,105,95,340,Southern Star Stationers
STA-0016,File Folder Pack of 10,Delhi North DC,29,120,91,380,Coromandel Traders
HOK-0021,Dinner Plate Set 6pc,Bengaluru Tech Park WH,25,115,90,460,Coromandel Traders
ELE-0005,HDMI Cable 2m,Chennai Central DC,8,95,87,380,Deccan Wholesale Co
ELE-0007,Laptop Stand,Bengaluru Tech Park WH,19,105,86,340,Deccan Wholesale Co
PCA-0035,Toothpaste 150g,Chennai Central DC,23,100,77,240,Gangetic Distributors
HOK-0017,Steel Water Bottle 1L,Delhi North DC,29,90,61,360,Western Ghats Supplies
TOH-0027,Measuring Tape 5m,Chennai Central DC,19,75,56,300,Gangetic Distributors
STA-0012,Stapler,Delhi North DC,22,75,53,300,Southern Star Stationers
HOK-0021,Dinner Plate Set 6pc,Chennai Central DC,68,115,47,460,Coromandel Traders


In [0]:
total_rows     = master.count()
expected_rows  = products.count() * warehouses.select("warehouse_id").distinct().count()
negative_stock = master.filter("quantity_on_hand < 0").count()
dup_keys       = total_rows - master.select("product_id", "warehouse_id").distinct().count()

print("rows in master inventory :", total_rows)
print("expected rows            :", expected_rows, "(products x warehouses)")
print("negative stock rows      :", negative_stock)
print("duplicate product+warehouse rows:", dup_keys)
assert negative_stock == 0 and dup_keys == 0
print("Master inventory checks passed")

rows in master inventory : 240
expected rows            : 240 (products x warehouses)
negative stock rows      : 0
duplicate product+warehouse rows: 0
Master inventory checks passed


In [0]:

(master.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(TABLE_NAME))
print("Delta table saved:", TABLE_NAME)

Delta table saved: workspace.default.master_inventory


In [0]:

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

master.toPandas().to_csv(f"{OUTPUT_DIR}/master_inventory.csv", index=False)
reorder_list.toPandas().to_csv(f"{OUTPUT_DIR}/reorder_list.csv", index=False)

warehouse_summary = spark.sql("""
    SELECT warehouse_name, city,
           SUM(quantity_on_hand)                          AS total_units,
           ROUND(SUM(stock_value), 2)                     AS total_stock_value,
           SUM(CASE WHEN reorder_flag THEN 1 ELSE 0 END)  AS items_to_reorder,
           SUM(CASE WHEN stock_status = 'OVERSTOCK' THEN 1 ELSE 0 END) AS overstocked_items
    FROM master_inventory
    GROUP BY warehouse_name, city
    ORDER BY warehouse_name
""")
warehouse_summary.toPandas().to_csv(f"{OUTPUT_DIR}/warehouse_summary.csv", index=False)

print("CSV files saved in", OUTPUT_DIR)
display(dbutils.fs.ls(OUTPUT_DIR))

CSV files saved in /Volumes/workspace/default/inventory_vol/gold


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/inventory_vol/gold/master_inventory.csv,master_inventory.csv,36377,1794691408000
dbfs:/Volumes/workspace/default/inventory_vol/gold/reorder_list.csv,reorder_list.csv,5852,1794691409000
dbfs:/Volumes/workspace/default/inventory_vol/gold/warehouse_summary.csv,warehouse_summary.csv,311,1794691410000


In [0]:
saved = spark.table(TABLE_NAME)
print("rows in Delta table:", saved.count())
display(spark.sql(f"DESCRIBE DETAIL {TABLE_NAME}"))      


rows in Delta table: 240


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,9f5b866d-a7ae-4e7d-900e-966161cd0fda,workspace.default.master_inventory,null,,2026-11-14T21:23:11.359Z,2026-11-14T21:23:14.000Z,List(),List(),1,10234,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
display(spark.sql(f"""
    SELECT warehouse_name,
           SUM(quantity_on_hand) AS total_units,
           SUM(CASE WHEN reorder_flag THEN 1 ELSE 0 END) AS items_to_reorder,
           SUM(CASE WHEN stock_status = 'OVERSTOCK' THEN 1 ELSE 0 END) AS overstocked_items
    FROM {TABLE_NAME}
    GROUP BY warehouse_name
    ORDER BY warehouse_name
"""))

warehouse_name,total_units,items_to_reorder,overstocked_items
Bengaluru Tech Park WH,14470,3,17
Chennai Central DC,7101,21,4
Delhi North DC,10755,6,7
Hyderabad South Hub,10591,2,6
Mumbai West Hub,14285,4,17


In [0]:
display(spark.sql(f"""
    SELECT category,
           SUM(CASE WHEN reorder_flag THEN 1 ELSE 0 END) AS items_to_reorder,
           ROUND(SUM(stock_value), 2) AS stock_value
    FROM {TABLE_NAME}
    GROUP BY category
    ORDER BY items_to_reorder DESC
"""))

category,items_to_reorder,stock_value
Sports & Fitness,7,5785860
Tools & Hardware,7,5564107
Home & Kitchen,6,8677973
Personal Care,6,3422462
Electronics,5,10719353
Stationery,5,3570345
